In [1]:
import pandas as pd
df = pd.read_csv("hf://datasets/opensporks/resumes/Resume/Resume.csv")
df.head()
print(len(df))

2484


### Structure
- SKillNER: extract skills directly from resume text

- TFIDF + LR: predict skills or categories using statistical patterns in text

### SkillNER Baseline Model

In [3]:
import re
import pandas as pd
from tqdm import tqdm

import spacy
from spacy.matcher import PhraseMatcher
from skillNer.general_params import SKILL_DB
from skillNer.skill_extractor_class import SkillExtractor

In [4]:
# clean over resume text
def clean_resume_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["resume_clean"] = df["Resume_str"].apply(clean_resume_text)

In [5]:
nlp = spacy.load("en_core_web_lg")
skill_extractor = SkillExtractor(nlp, SKILL_DB, PhraseMatcher)

def extract_skills_skillner_fast(text):
    if not text or len(text.strip()) == 0:
        return []
    
    try:
        annotations = skill_extractor.annotate(text)
        results = annotations.get("results", {})
        full_matches = results.get("full_matches", [])
        
        extracted = []
        
        # SkillNER output can be list or dict depending on version
        if isinstance(full_matches, dict):
            iterable = full_matches.values()
        else:
            iterable = full_matches
        
        for match in iterable:
            if isinstance(match, dict):
                skill = (
                    match.get("doc_node_value")
                    or match.get("doc_node_name")
                    or match.get("skill_id")
                )
                if skill:
                    extracted.append(str(skill).lower())
        
        # deduplicate while preserving order
        return list(dict.fromkeys(extracted))
    
    except Exception:
        return []

loading full_matcher ...
loading abv_matcher ...
loading full_uni_matcher ...
loading low_form_matcher ...
loading token_matcher ...


In [6]:
# Test on small samples
test_df = df.head(20).copy()

test_skills = []
for text in tqdm(test_df["resume_clean"], total=len(test_df)):
    test_skills.append(extract_skills_skillner_fast(text))

test_df["skills_skillner"] = test_skills

print(test_df[["ID", "Category", "skills_skillner"]].head(10))
print(test_df["skills_skillner"].apply(len).describe())

  0%|          | 0/20 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/skillNer/utils.py:99: UserWarning: [W008] Evaluating Token.similarity based on empty vectors.
  vec_similarity = token1.similarity(token2)
100%|██████████| 20/20 [02:47<00:00,  8.40s/it]

         ID Category                                    skills_skillner
0  16852973       HR  [customer service, service management, custome...
1  22323967       HR  [group policy, process improvement, visual com...
2  33176873       HR  [benefit administration, policy development, w...
3  27018550       HR  [customer service, customer satisfaction, proc...
4  17812897       HR  [employee relation, benefit administration, pe...
5  11592605       HR  [microsoft office, time management, self start...
6  25824789       HR  [conflict resolution, record management, polic...
7  15375009       HR  [human resource management, resource managemen...
8  11847784       HR  [middle management, training and development, ...
9  32896934       HR  [employee engagement, corporate social respons...
count    20.00000
mean     21.05000
std       7.16332
min      10.00000
25%      17.50000
50%      20.00000
75%      23.50000
max      41.00000
Name: skills_skillner, dtype: float64


In [7]:
skills_list = []
for text in tqdm(df["resume_clean"], total=len(df)):
    skills_list.append(extract_skills_skillner_fast(text))

df["skills_skillner"] = skills_list

print(df[["ID", "Category", "skills_skillner"]].head(10))
print(df["skills_skillner"].apply(len).describe())

# df.to_csv("resume_skills_skillner_fast.csv", index=False)

  0%|          | 0/2484 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/skillNer/utils.py:99: UserWarning: [W008] Evaluating Token.similarity based on empty vectors.
  vec_similarity = token1.similarity(token2)
100%|██████████| 2484/2484 [9:06:37<00:00, 13.20s/it]    

         ID Category                                    skills_skillner
0  16852973       HR  [customer service, service management, custome...
1  22323967       HR  [group policy, process improvement, visual com...
2  33176873       HR  [benefit administration, policy development, w...
3  27018550       HR  [customer service, customer satisfaction, proc...
4  17812897       HR  [employee relation, benefit administration, pe...
5  11592605       HR  [microsoft office, time management, self start...
6  25824789       HR  [conflict resolution, record management, polic...
7  15375009       HR  [human resource management, resource managemen...
8  11847784       HR  [middle management, training and development, ...
9  32896934       HR  [employee engagement, corporate social respons...
count    2484.000000
mean       19.380837
std        11.436031
min         0.000000
25%        12.000000
50%        18.000000
75%        25.000000
max        98.000000
Name: skills_skillner, dtype: float64


### TF-IDF + Multi-label Logistic Regression

In [9]:
import warnings
warnings.filterwarnings("ignore")

df_model = df[df["skills_skillner"].apply(lambda x: isinstance(x, list) and len(x) > 0)].copy()
print("Rows used for model:", len(df_model))

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df_model["skills_skillner"])

from sklearn.model_selection import train_test_split

X_train_text, X_test_text, Y_train, Y_test = train_test_split(
    df_model["resume_clean"],
    Y,
    test_size=0.2,
    random_state=42
)

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

clf = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        solver="liblinear"
    )
)

clf.fit(X_train, Y_train)

Y_pred = clf.predict(X_test)

Rows used for model: 2439


In [ ]:
# Examine model performance

from sklearn.metrics import classification_report
print(classification_report(Y_test, Y_pred, target_names=mlb.classes_))

                                                                                     precision    recall  f1-score   support

                                                                       3d animation       0.00      0.00      0.00         1
                                                                             3d art       0.00      0.00      0.00         0
                                                                        3d modeling       0.00      0.00      0.00         3
                                                                        3d printing       0.00      0.00      0.00         1
                                                                       3d rendering       0.00      0.00      0.00         0
                                                                           3d touch       0.00      0.00      0.00         0
                                                                   3d visualization       0.00      0.00      0.00         2

In [11]:
from sklearn.metrics import f1_score

micro_f1 = f1_score(Y_test, Y_pred, average="micro")
macro_f1 = f1_score(Y_test, Y_pred, average="macro")

print("Micro F1:", micro_f1)
print("Macro F1:", macro_f1)

Micro F1: 0.08998117134079874
Macro F1: 0.002550104796700703


In [12]:
def decode_labels(binary_row):
    return [mlb.classes_[i] for i,v in enumerate(binary_row) if v==1]

for i in range(5):
    print("Resume text:", X_test_text.iloc[i][:200])
    print("True skills:", decode_labels(Y_test[i]))
    print("Predicted:", decode_labels(Y_pred[i]))
    print()

Resume text: community advocate summary dedicated and focused community advocate who excels at prioritizing, completing multiple tasks simultaneously and following through to achieve project goals. seeking a role 
True skills: ['behavioral science', 'constructive feedback', 'cpr', 'customer service', 'detail orient', 'first aid', 'organizational skill', 'time management']
Predicted: ['customer service']

Resume text: engineering teacher professional summary to obtain a challenging position in the field of engineering and to work within a team environment, where i can contribute my skills and experience to a client
True skills: ['achievement orient', 'adobe acrobat', 'air quality', 'apply science', 'autodesk inventor', 'autodesk revit', 'blueprint reading', 'building code', 'capital budgeting', 'civil engineering', 'css', 'design technology', 'feasibility study', 'gis', 'gps', 'html', 'mechanical engineering', 'microsoft office', 'personal computer', 'plumbing code', 'professional engin